# Fine-Tuning LLMs
Adapt a pre-trained model to your specific task with minimal data.

In [ ]:
import micropip
await micropip.install(['scikit-learn','matplotlib','numpy'])
print('Ready!')

## When to Fine-Tune

| Approach | When | Data | Cost |
|---|---|---|---|
| Prompting | General tasks | None | API |
| RAG | Domain Q&A | Documents | Low |
| Fine-tuning | Custom style, edge cases | 100-10K examples | Medium |
| Pre-training | New domain from scratch | Billions of tokens | Very high |

**Rule:** Try prompting -> RAG -> fine-tune.

In [ ]:
# TF-IDF + LogReg baseline
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report
from sklearn.model_selection import train_test_split
reviews = [
    'Absolutely brilliant film. The acting was superb and story compelling.',
    'Complete waste of time. Boring from start to finish.',
    'A masterpiece of modern cinema. Highly recommended.',
    'Terrible script, poor direction. Very disappointed.',
    'Outstanding performances. Moved me deeply.',
    'Dreadful film. Left halfway through.',
    'Heartwarming and funny. Perfect family entertainment.',
    'Overrated and dull. Fell asleep several times.',
    'Exceptional storytelling. One of the best films this year.',
    'Awful. Nothing redeeming about this production.',
]
labels = [1,0,1,0,1,0,1,0,1,0]
X_tr,X_te,y_tr,y_te = train_test_split(reviews,labels,test_size=0.3,random_state=42)
vec = TfidfVectorizer(ngram_range=(1,2))
clf = LogisticRegression().fit(vec.fit_transform(X_tr),y_tr)
print(f'TF-IDF baseline accuracy: {accuracy_score(y_te,clf.predict(vec.transform(X_te))):.3f}')

## DistilBERT Fine-Tuning (Kaggle GPU)

```python
from transformers import DistilBertTokenizerFast, DistilBertForSequenceClassification, Trainer, TrainingArguments
from datasets import Dataset

tokenizer = DistilBertTokenizerFast.from_pretrained('distilbert-base-uncased')

def tokenize(batch):
    return tokenizer(batch['text'], truncation=True, padding=True, max_length=128)

dataset = Dataset.from_dict({'text': reviews, 'label': labels}).map(tokenize, batched=True)
train_ds, eval_ds = dataset.train_test_split(test_size=0.2).values()
model = DistilBertForSequenceClassification.from_pretrained('distilbert-base-uncased', num_labels=2)
args = TrainingArguments(output_dir='./model', num_train_epochs=3,
                         per_device_train_batch_size=16, learning_rate=2e-5)
trainer = Trainer(model=model, args=args, train_dataset=train_ds, eval_dataset=eval_ds)
trainer.train()
```

## LoRA -- Parameter-Efficient Fine-Tuning

```python
# pip install peft
from peft import LoraConfig, get_peft_model

config = LoraConfig(r=8, lora_alpha=32, lora_dropout=0.1,
                    target_modules=['q_lin', 'v_lin'])
model = get_peft_model(model, config)
model.print_trainable_parameters()
# trainable params: 147,456 / all params: 66,955,010 = 0.22%
# Only 0.22% of weights train -- 99.78% frozen!
```

**LoRA math:** W_new = W_pretrained + B*A
- W frozen, B(m x r) and A(r x n) train
- r=8 is typical. Lower r = smaller update, less forgetting.

In [ ]:
# Compare parameter counts: full fine-tune vs LoRA
model_params = {
    'DistilBERT base': 66_955_010,
    'BERT base':       110_000_000,
    'LLaMA-7B':        7_000_000_000,
    'LLaMA-70B':      70_000_000_000,
}
for model_name, params in model_params.items():
    lora_params = int(params * 0.0022)  # ~0.22% for r=8
    print(f'{model_name:<20}: {params/1e6:>8.1f}M total | {lora_params/1e6:.2f}M LoRA trainable')

---
**Your turn:** Research: what is the difference between LoRA and QLoRA? When would you choose QLoRA?